# Estrategia de ofertas FDS 3–5 julio 2026 — Pricing Jüsto
**Autor:** Joaquín Bolaños · Data Science, Pricing · Dark stores 9 y 14
**Insumos (únicos):** `Oportunidad.xlsx` (universo y atributos por SKU-tienda) y `Descuentos_comerciales.xlsx` (blacklist y calendario de campañas).
**Entregable:** `Estrategia_Pricing_FDS_Julio.xlsx` — estrategia por SKU con mecánica, precio, día de ejecución, marca de temporada Mundial y proyección económica con escenarios.

## Encargo
Diseñar la estrategia de ofertas del fin de semana **sin descuentos comerciales**: cada oferta se financia con el colchón de margen del propio SKU. Restricciones: piso de pure margin del 22% post-oferta, tope autónomo del 30% de descuento (más requiere Vo.Bo. de Comercial), protección del posicionamiento de precio, y mecánicas permitidas SPON / BNSP / BNSDP / 2x1 / 3x2 / 4x3 / 5x4 / 6x5.

## Principios de diseño del modelo económico
Una proyección de promociones falla de dos maneras clásicas, y este modelo se construye explícitamente contra ambas:

1. **El descuento no lo paga el 100% de las unidades.** En mecánicas de volumen, solo la fracción de compradores que alcanza el umbral (la **tasa de redención** $r$) recibe el precio rebajado; el resto paga precio lleno. Un modelo que aplica el descuento a todas las unidades sobreestima el costo de la promoción y proyecta GMV y utilidad artificialmente bajos.
2. **La demanda no responde a la elasticidad de lista, responde a la oferta exhibida.** Un "3 x \$99" comunicado en la app genera una respuesta mayor que una baja silenciosa de precio de góndola del mismo porcentaje: hay efecto de visibilidad, ancla psicológica y sensación de oportunidad. El modelo lo captura con un **multiplicador promocional** sobre la elasticidad base.

A esos dos principios se suma un tercero de disciplina: **ninguna oferta se publica si su utilidad incremental proyectada es negativa en el escenario base**. El piso del 22% protege el margen unitario; este guardrail económico protege el resultado total.

## Mapa del pipeline
1. **Parámetros** — todas las perillas del modelo en una celda.
2. **Carga y parsing** — limpieza del universo y separación de `SKU`/`Nombre`.
3. **Cruce comercial** — blacklist fija + campañas cuyo rango cruza el FDS (unión de dos hojas operativas, menos las eliminadas); las filas se marcan, no se borran.
4. **Fuente de verdad del margen** — `MARGEN` se usa directa; el precio neto solo convierte utilidad a pesos.
5. **Elegibilidad** — margen ≥ 22% y elasticidad utilizable.
6. **Tema Mundial** — detección por patrón sobre el nombre + impulso de prioridad.
7. **Descuento máximo** $d_{max}$ — álgebra sobre `MARGEN`.
8. **Score y descuento objetivo** — elasticidad × rotación + share + Mundial, con castigo de posicionamiento.
9. **Asignación de mecánica** — cascada multibuy → BNSDP/BNSP/SPON según ticket.
10. **Validación del piso** — margen post-oferta sobre el descuento efectivo real.
11. **Modelo económico** — redención, multiplicador promocional, GMV y utilidad mezclados.
12. **Guardrail económico** — se descartan ofertas con utilidad incremental proyectada < 0.
13. **Día de ejecución** — viernes / sábado / domingo por perfil.
14. **Escenarios** — sensibilidad conservador / base / optimista.
15. **Resultados, exportación y cierre.**

## Glosario de notación

| Símbolo | Nombre | Significado |
|---|---|---|
| $P$ | Precio de góndola | `PRECIO JUSTO`, con impuestos incluidos; no cambia con estas mecánicas |
| $m$ | Pure margin | `MARGEN`/100 — margen sobre ingreso neto. **Única fuente de verdad** |
| $P_{neto}$ | Precio neto | $P/[(1+IVA)(1+IEPS)]$; solo convierte utilidad a pesos |
| $d$ | Descuento efectivo | Rebaja por unidad **cuando se activa la mecánica** (la profundidad exhibida) |
| $d_{max}$ | Descuento máximo | Colchón del SKU: $1-(1-m)/0{,}78$ |
| $d_{target}$ | Descuento objetivo | $\min(d_{max}, TOPE)$ |
| $P_{eff}$ | Precio efectivo | $P(1-d)$: lo que paga quien activa la mecánica (`PRECIO_OFERTA`) |
| $m_d$ | Margen con descuento | $1-(1-m)/(1-d)$: margen de las unidades vendidas con la mecánica activa |
| $r$ | Tasa de redención | Fracción de las unidades que se venden con la mecánica activa |
| $u$ | Uplift de demanda | Crecimiento fraccional de unidades: $|E| \times MULT \times d$ |
| $E$ | Elasticidad base | Sensibilidad de lista, supuesta por tag (negativa) |
| $MULT$ | Multiplicador promocional | Cuánto amplifica la exhibición de la oferta la respuesta de lista |
| $Q_0, Q_1$ | Demanda base / con promo | Unidades/día: $Q_1 = Q_0(1+u)$ |

Dos identidades del modelo mezclado que hacen todo el trabajo (derivadas en la sección 11): el ingreso por unidad vendida es $P(1 - r\,d)$ y la utilidad por unidad vendida es $P_{neto}(m - r\,d)$.

## 1 · Parámetros del modelo
Cada supuesto vive aquí, con su justificación:

- **Redención por mecánica** ($r$): qué fracción de las unidades del día se vende con la mecánica activa. A mayor umbral de unidades, menor redención: las mecánicas de 2 unidades convierten más que las de 4–6. Valores típicos de retail de conveniencia, declarados y ajustables.
- **Elasticidad base por tag** ($E$): magnitudes estándar para consumo masivo. Solo dimensionan la respuesta; la selección de SKUs no depende de ellas.
- **Multiplicador promocional** ($MULT = 2{,}0$): la literatura de promociones en retail ubica la respuesta a ofertas exhibidas entre 1,5 y 3 veces la elasticidad de lista; se toma el centro del rango como escenario base y la sección 14 lo somete a sensibilidad (0,6× / 1,0× / 1,4×).
- **Demanda base por rotación** ($Q_0$, unidades/día por SKU-tienda): los insumos autorizados no traen ventas, así que $Q_0$ es un supuesto de dimensionamiento declarado. Importa la honestidad de la lectura: los **porcentajes** de uplift por SKU no dependen de $Q_0$; los **pesos totales** escalan linealmente con él y se reportan como orden de magnitud.

In [1]:
import pandas as pd
import numpy as np
import re, os

# ---------- Archivos (únicos insumos autorizados) ----------
RUTA_OPORTUNIDAD = 'Oportunidad.xlsx'
RUTA_DESCUENTOS  = 'Descuentos_comerciales.xlsx'
RUTA_SALIDA      = 'Estrategia_Pricing_FDS_Julio.xlsx'

# ---------- Fin de semana objetivo ----------
WEEKEND_INICIO = pd.Timestamp('2026-07-03')   # viernes
WEEKEND_FIN    = pd.Timestamp('2026-07-05')   # domingo

# ---------- Reglas de margen ----------
PISO_MARGEN    = 0.22
TOPE_GUARDRAIL = 0.30
DESC_MINIMO    = 0.05

# ---------- Priorización ----------
TOPES_SCORE        = [(7, 0.25), (5, 0.20), (3, 0.15), (2, 0.10)]
CASTIGO_MAS_BARATO = 0.05
BOOST_MUNDIAL      = 1
TOLERANCIA_CASCADA = 0.02

# ---------- Mecánicas flexibles ----------
TICKET_ALTO = 150
TICKET_BAJO = 60

# ---------- Modelo económico ----------
MULT_PROMO = 2.0                                  # amplificación promocional del escenario base
ELASTICIDAD_POR_TAG = {'ALTAMENTE ELASTICO': -2.2, 'ELASTICO': -1.5, 'POCO ELASTICO': -0.8}
REDENCION = {'multibuy_grande': 0.35,             # 4x3 / 5x4 / 6x5 (umbral de 4-6 unidades)
             'pack_3': 0.40,                      # BNSP 3 unidades
             'umbral_2': 0.55}                    # SPON / BNSDP de 2 unidades
Q0_POR_ROTACION = {'Alta rotacion': 20.0, 'Rotacion normal': 6.0,
                   'Baja rotacion': 1.5, 'Sin rotacion': 0.3}   # unidades/día (dimensionamiento)
ESCENARIOS = {'Conservador': 0.6, 'Base': 1.0, 'Optimista': 1.4}  # factor sobre el uplift

pd.set_option('display.max_colwidth', 60)

## 2 · Carga y parsing del universo
Se retiran la fila de notas del final del export y las filas vacías, y se separa `SKU + Nombre` en `SKU` (entero, la llave de cruce) y `Nombre` (texto, insumo de la detección Mundial). La llave de análisis es **tienda × SKU**: un mismo producto puede tener costo, precio y comportamiento distintos por dark store, y la estrategia se decide por combinación.

La nota del propio archivo documenta el pre-filtrado de la fuente: sin marca propia, sin blacklist, sin promoción activa hoy, solo SKUs prendidos, grupo de recomendación *OPORTUNIDAD*. La sección 3 agrega lo que ese filtro no puede ver: campañas que **inician** durante el fin de semana.

In [2]:
# Se carga el export y se limpia
df = pd.read_excel(RUTA_OPORTUNIDAD, sheet_name='Export')
print(f"Filas originales del export: {len(df)}")

df = df[df['SKU + Nombre'].notna() & df['COSTO'].notna()].copy()
df['STORE_ID'] = pd.to_numeric(df['STORE_ID'], errors='coerce')
df = df[df['STORE_ID'].notna()].copy()
df['STORE_ID'] = df['STORE_ID'].astype(int)

# Se separa la llave: 'SKU + Nombre' -> SKU entero y Nombre
df = df[df['SKU + Nombre'].astype(str).str.match(r'^\d+\s*-\s*.+')].copy()
df[['SKU', 'Nombre']] = df['SKU + Nombre'].str.split(' - ', n=1, expand=True)
df['SKU'] = df['SKU'].astype(int)

print(f"Filas válidas (tienda x SKU): {len(df)} | Tiendas: {sorted(df['STORE_ID'].unique())}")
df[['STORE_ID','SKU','Nombre','MARGEN','PRECIO JUSTO','TAG_ROTACION','TAG ELAS','TAG_SHARE']].head(3)

/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Filas originales del export: 2474
Filas válidas (tienda x SKU): 2472 | Tiendas: [np.int64(9), np.int64(14)]


,STORE_ID,SKU,Nombre,MARGEN,PRECIO JUSTO,TAG_ROTACION,TAG ELAS,TAG_SHARE
0,9,24633,Frambuesas Franuí Bañadas Chocolate Amargo y Blanco 150g,22.92,139.01,Alta rotacion,ALTAMENTE ELASTICO,ALTO SHARE
1,9,11365,Bolsas Reutilizables para Sándwich Ziploc 50 pzs,22.12,58.22,Alta rotacion,ELASTICO,ALTO SHARE
2,9,24002,Barra de Proteína Wild Protein Chocolate 225g,29.69,149.00,Alta rotacion,ELASTICO,ALTO SHARE


## 3 · Cruce comercial: blacklist y campañas que cruzan el FDS
El archivo de descuentos trae siete hojas; cuatro son operativas para este ejercicio:

- **`Black list`**: exclusión fija por `ItemId` + `AreaId`, sin ventana de fechas.
- **`DEscuentos comerciales`** y **`Historico`** (esta última filtrada a estatus *Cargado*): dos registros de campañas con `FechaInicio`/`FechaFin`. Se toma la **unión** de ambas y se conservan las campañas cuyo rango **cruza el fin de semana objetivo** — la campaña toca el FDS si inicia antes del fin y termina después del inicio.
- **`Eliminacion de campañas`**: bajas confirmadas; sus pares se **restan** del conjunto, porque una campaña eliminada ya no compite.

Las hojas `Ejemplo`, `Horarios de carga` y `Hoja 3` son plantillas o metadatos y se ignoran deliberadamente.

**Dos decisiones de diseño:** las filas afectadas se **marcan con su motivo, nunca se borran** (el entregable contiene la decisión sobre cada SKU del universo), y los identificadores pasan por coerción numérica antes del cruce porque `ItemId` llega con tipos mixtos entre hojas. El sentido de negocio: un SKU con campaña comercial activa en el FDS no debe recibir además una oferta de pricing — el descuento se duplicaría y el margen real quedaría por debajo de lo simulado.

In [3]:
df['MOTIVO_SIN_OFERTA'] = ''

def pares_limpios(tabla, cols=('ItemId', 'AreaId')):
    """Convierte ItemId/AreaId a enteros con coerción numérica y devuelve el set de pares."""
    t = tabla[list(cols)].apply(pd.to_numeric, errors='coerce').dropna().astype(int)
    return set(map(tuple, t.values))

def campanas_en_fds(hoja, filtro_cargado=False):
    """Carga una hoja de campañas y devuelve las que cruzan el FDS objetivo."""
    c = pd.read_excel(RUTA_DESCUENTOS, sheet_name=hoja)
    if filtro_cargado and 'Estatus' in c.columns:
        c = c[c['Estatus'].astype(str).str.strip().str.lower() == 'cargado']
    c = c[['ItemId', 'AreaId', 'FechaInicio', 'FechaFin']].copy()
    c[['ItemId', 'AreaId']] = c[['ItemId', 'AreaId']].apply(pd.to_numeric, errors='coerce')
    c['FechaInicio'] = pd.to_datetime(c['FechaInicio'], errors='coerce')
    c['FechaFin']    = pd.to_datetime(c['FechaFin'], errors='coerce')
    c = c.dropna()
    return c[(c['FechaInicio'] <= WEEKEND_FIN) & (c['FechaFin'] >= WEEKEND_INICIO)]

black_pares = pares_limpios(pd.read_excel(RUTA_DESCUENTOS, sheet_name='Black list'))
promo_pares = (pares_limpios(campanas_en_fds('DEscuentos comerciales'))
               | pares_limpios(campanas_en_fds('Historico', filtro_cargado=True)))
promo_pares -= pares_limpios(campanas_en_fds('Eliminacion de campañas'))

pares = list(zip(df['SKU'], df['STORE_ID']))
en_black = pd.Series([p in black_pares for p in pares], index=df.index)
en_promo = pd.Series([p in promo_pares for p in pares], index=df.index) & ~en_black
df.loc[en_black, 'MOTIVO_SIN_OFERTA'] = 'Blacklist comercial'
df.loc[en_promo, 'MOTIVO_SIN_OFERTA'] = 'Campaña comercial cruza el FDS'

print(f"Pares en blacklist: {len(black_pares)} | Pares de campaña que cruzan el FDS: {len(promo_pares)}")
print(f"Filas del universo marcadas -> blacklist: {en_black.sum()} | campaña en FDS: {en_promo.sum()}")

Pares en blacklist: 171 | Pares de campaña que cruzan el FDS: 953
Filas del universo marcadas -> blacklist: 0 | campaña en FDS: 7


**Interpretación.** Que la blacklist marque cero filas del universo **valida el pre-filtrado de la fuente** (el export ya venía sin blacklist). Las filas marcadas por campaña son el hallazgo real del cruce: campañas que inician durante el fin de semana, invisibles para el filtro "sin promoción activa *hoy*". Sin este paso habrían recibido doble descuento.

## 4 · Fuente de verdad del margen
La columna `MARGEN` es la definición oficial del pure margin: margen sobre el precio neto de impuestos con estructura **secuencial**, $P_{neto} = P/[(1+IVA)(1+IEPS)]$. Toda la matemática del piso se hace por álgebra directa sobre $m$ = `MARGEN`/100 — sin reconstruir el margen desde costo e impuestos, que es la fuente clásica de discrepancias contables. El precio neto se calcula **una sola vez** y con un único papel: convertir la utilidad proyectada a pesos en la sección 11.

La propiedad que habilita todo el pipeline: como los impuestos son proporcionales al precio, un descuento $d$ reduce el ingreso bruto y el neto en el mismo factor $(1-d)$ mientras el costo no cambia; la fracción del ingreso que se va en costo pasa de $(1-m)$ a $(1-m)/(1-d)$, y el margen con la mecánica activa queda:

$$m_d = 1 - \frac{1-m}{1-d}$$

Verificación de cordura con números redondos: $m = 40\%$ y $d = 23{,}1\%$ dan $m_d = 1 - 0{,}60/0{,}769 = 22{,}0\%$ — exactamente el piso, como debe ser (ese $d$ es el colchón completo de un SKU al 40%).

In [4]:
# Margen como única fuente de verdad; precio neto solo para pesos de utilidad (sección 11)
m = df['MARGEN'] / 100
df['PRECIO_NETO'] = (df['PRECIO JUSTO']
                     / (1 + df['IVA'].fillna(0) / 100)
                     / (1 + df['IEPS'].fillna(0) / 100))
print(f"Margen del universo -> mediana: {df['MARGEN'].median():.1f}% | "
      f"p25: {df['MARGEN'].quantile(.25):.1f}% | p75: {df['MARGEN'].quantile(.75):.1f}%")

Margen del universo -> mediana: 25.0% | p25: 21.5% | p75: 28.9%


## 5 · Elegibilidad
Dos filtros explícitos, aplicados solo sobre filas aún sin motivo (precedencia: comercial > margen > elasticidad):

- **Margen base < 22%**: el SKU ya está en el piso o debajo; no existe colchón que financiar.
- **INELASTICO o SIN DATOS**: descontar donde la demanda no responde — o donde no hay evidencia de que responda — es ceder margen sin volumen a cambio. `POCO ELASTICO` sí permanece: responde poco, pero responde, y el score le dosifica la agresividad.

Las filas excluidas permanecen en el entregable con su motivo visible.

In [5]:
sin_motivo = df['MOTIVO_SIN_OFERTA'] == ''

m_bajo = (df['MARGEN'] < PISO_MARGEN * 100) & sin_motivo
df.loc[m_bajo, 'MOTIVO_SIN_OFERTA'] = 'Margen base < 22%'

elas_out = df['TAG ELAS'].isin(['INELASTICO', 'SIN DATOS']) & sin_motivo & ~m_bajo
df.loc[elas_out, 'MOTIVO_SIN_OFERTA'] = 'Inelástico / sin datos de elasticidad'

df['ELEGIBLE'] = df['MOTIVO_SIN_OFERTA'] == ''
print("Excluidos por causa:")
print(df.loc[~df['ELEGIBLE'], 'MOTIVO_SIN_OFERTA'].value_counts().to_string())
print(f"\nUniverso elegible: {df['ELEGIBLE'].sum()} de {len(df)} filas")

Excluidos por causa:
MOTIVO_SIN_OFERTA
Margen base < 22%                        654
Inelástico / sin datos de elasticidad    572
Campaña comercial cruza el FDS             7

Universo elegible: 1239 de 2472 filas


## 6 · Tema Mundial (valor agregado de temporada)
El fin de semana objetivo cae en **octavos de final del Mundial 2026 con México como anfitrión**. El catálogo no tiene merchandising con licencia (cero coincidencias para "mundial", "fútbol", "FIFA", "selección"), así que se usa un proxy de consumo de "ver el partido": cerveza, refrescos, botanas saladas, totopos, palomitas y salsas picantes, detectados por patrón sobre el nombre y depurado a mano contra falsos positivos. El uso es doble: **+1 punto de score** (más agresividad dentro de lo que el margen permita) y **sábado como día de ejecución** (sección 13), donde el contexto multiplica la demanda.

In [6]:
PATRON_MUNDIAL = re.compile(
    r"\b(?:"
    r"cerveza|refresco|"
    r"papas?\s+(?:fritas?|pringles|ruffles)|totopos?|nachos?|chicharr\w*|"
    r"palomitas|botanas?|"
    r"habanero|valentina|cholula|clamato|b[uú]falo"
    r")\b",
    re.IGNORECASE,
)
df['TEMA_MUNDIAL'] = df['Nombre'].str.contains(PATRON_MUNDIAL, na=False)
print(f"Filas afines al Mundial: {df['TEMA_MUNDIAL'].sum()} ({df[df['TEMA_MUNDIAL']]['SKU'].nunique()} SKUs únicos)")

Filas afines al Mundial: 148 (85 SKUs únicos)


## 7 · Descuento máximo $d_{max}$
El colchón de cada SKU: imponiendo el piso $m_d \geq 0{,}22$ en la identidad de la sección 4 y despejando $d$:

$$1 - \frac{1-m}{1-d} \geq 0{,}22 \;\Rightarrow\; d \leq 1 - \frac{1-m}{0{,}78} \;=\; d_{max}$$

**Lectura de cada término:** $1-m$ es la fracción del ingreso que hoy se va en costo; $0{,}78$ es la fracción máxima que puede irse en costo con el margen exactamente en el piso (viene de `PISO_MARGEN` — si el piso cambiara, el código se ajusta solo). Invirtiendo ($m = 0{,}22 + 0{,}78\,d$) se obtiene el margen mínimo que exige cada mecánica: 6x5 → 35,0%, 5x4 → 37,6%, 4x3 → 41,5%, 3x2 → 48,0%, 2x1 → 61,0%. Con el tope autónomo del 30%, el 2x1 y el 3x2 quedan fuera del menú sin Vo.Bo.: la mecánica con nombre más agresiva ejecutable es el 4x3.

In [7]:
df['DMAX'] = (1 - (1 - m) / (1 - PISO_MARGEN)).clip(lower=0, upper=TOPE_GUARDRAIL)
df.loc[~df['ELEGIBLE'], 'DMAX'] = 0.0

eleg = df[df['ELEGIBLE']]
print(f"d_max de los elegibles -> mediana: {eleg['DMAX'].median():.1%} | "
      f"soportan 6x5: {(eleg['DMAX'] >= 1/6).sum()} | soportan 4x3: {(eleg['DMAX'] >= 0.25).sum()}")

d_max de los elegibles -> mediana: 5.8% | soportan 6x5: 84 | soportan 4x3: 32


## 8 · Score y descuento objetivo

$$SCORE = W_{elas} \times W_{rot} + S_{share} + S_{mundial}$$

La multiplicación es deliberada: un cero en elasticidad **o** en rotación anula la prioridad, porque el uplift porcentual sin volumen base no mueve dinero, y el volumen sin respuesta al precio no se amplifica con descuento. $S_{share}$ (+1 con alto share) prioriza los SKUs que pesan en la venta; $S_{mundial}$ (+1) empuja los de temporada. El score se traduce a $TOPE$ (la fracción del colchón que se autoriza a usar) con la tabla de parámetros, y el **castigo de posicionamiento** resta un nivel a los SKUs que ya son 'Mas barato' en línea: la percepción de precio ya está comprada y pagarla dos veces es desperdicio.

$$d_{target} = \min(d_{max},\; TOPE)$$

El descuento final es el menor entre lo que el margen **permite** y lo que el perfil **amerita**; la oferta solo existe donde ambos dan espacio.

In [8]:
elas_w = {'ALTAMENTE ELASTICO': 3, 'ELASTICO': 2, 'POCO ELASTICO': 1, 'INELASTICO': 0, 'SIN DATOS': 0}
rot_w  = {'Alta rotacion': 3, 'Rotacion normal': 2, 'Baja rotacion': 1, 'Sin rotacion': 0}

df['W_ELAS'] = df['TAG ELAS'].map(elas_w).fillna(0)
df['W_ROT']  = df['TAG_ROTACION'].map(rot_w).fillna(0)
df['SCORE']  = (df['W_ELAS'] * df['W_ROT']
                + (df['TAG_SHARE'] == 'ALTO SHARE').astype(int)
                + df['TEMA_MUNDIAL'].astype(int) * BOOST_MUNDIAL)

def tope_por_score(s):
    """Devuelve la fracción máxima del colchón autorizada para un score dado."""
    for smin, tope in TOPES_SCORE:
        if s >= smin:
            return tope
    return 0.0

df['TOPE'] = df['SCORE'].apply(tope_por_score)
mas_barato = df['INDEX PRECIO LINEA'] == 'Mas barato'
df.loc[mas_barato, 'TOPE'] = (df.loc[mas_barato, 'TOPE'] - CASTIGO_MAS_BARATO).clip(lower=0)

df['D_TARGET'] = np.minimum(df['DMAX'], df['TOPE'])
df.loc[~df['ELEGIBLE'], 'D_TARGET'] = 0.0
print(f"Filas 'Mas barato' castigadas: {mas_barato.sum()} ({mas_barato.mean()*100:.0f}% del universo)")
print(f"Elegibles con d_target >= {DESC_MINIMO:.0%}: {(df['D_TARGET'] >= DESC_MINIMO).sum()}")

Filas 'Mas barato' castigadas: 1770 (72% del universo)
Elegibles con d_target >= 5%: 531


## 9 · Asignación de mecánica
Cascada de la más agresiva a la más suave. Multibuys con nombre primero (con **tolerancia**: si $d_{target}$ quedó a menos de 2 puntos del umbral y el margen alcanza — $d_{max} \geq$ umbral —, se sube, porque "6x5" comunica mejor que "-15% llevando 2"); luego mecánica flexible por ticket: **BNSDP** en pesos redondos para ticket alto (el ahorro en pesos comunica y controla el gasto), **BNSP** de paquete redondo para ticket bajo de alta rotación (precio psicológico de tráfico), **SPON** por defecto. Redondeos **siempre hacia abajo** — hacia arriba podrían cruzar $d_{max}$. Toda mecánica exige 2 o 3 unidades: el precio de góndola no cambia, quien compra una unidad paga completo, y el carrito crece.

In [9]:
UMBRALES_MULTIBUY = [(0.25, '4x3', 3, 4), (0.20, '5x4', 4, 5), (1/6, '6x5', 5, 6)]

def asignar(r):
    """Asigna mecánica, precio efectivo y descuento efectivo a cada fila."""
    d, dmax, p = r['D_TARGET'], r['DMAX'], r['PRECIO JUSTO']
    if d < DESC_MINIMO:
        return pd.Series(['Sin oferta', p, 0.0])
    for umbral, nombre, n_pag, n_llev in UMBRALES_MULTIBUY:
        if d >= umbral or ((umbral - d <= TOLERANCIA_CASCADA) and (dmax >= umbral)):
            return pd.Series([nombre, round(p * n_pag / n_llev, 2), round(umbral, 4)])
    d5 = np.floor(d * 20) / 20                       # múltiplo de 5% hacia abajo
    if d5 < DESC_MINIMO:
        return pd.Series(['Sin oferta', p, 0.0])
    if p >= TICKET_ALTO:
        ahorro = np.floor(d5 * p / 5) * 5            # pesos, múltiplo de $5 hacia abajo
        if ahorro < 5:
            return pd.Series(['Sin oferta', p, 0.0])
        return pd.Series([f'BNSDP: 2 uds, ahorra ${ahorro:.0f} c/u', round(p - ahorro, 2),
                          round(ahorro / p, 4)])
    if p <= TICKET_BAJO and r['W_ROT'] == 3:
        pack = np.floor(3 * p * (1 - d5))            # paquete de 3 a peso entero
        return pd.Series([f'BNSP: 3 x ${pack:.0f}', round(pack / 3, 2),
                          round(1 - pack / (3 * p), 4)])
    return pd.Series([f'SPON: 2 uds, -{d5*100:.0f}%', round(p * (1 - d5), 2), d5])

df[['ESTRATEGIA', 'PRECIO_OFERTA', 'DESC_EFECTIVO']] = df.apply(asignar, axis=1)
df['MECANICA'] = df['ESTRATEGIA'].str.split(':').str[0]
df['MECANICA'].value_counts()

MECANICA
Sin oferta    1941
SPON           316
BNSP           118
BNSDP           75
6x5             16
5x4              4
4x3              2
Name: count, dtype: int64

## 10 · Validación del piso
El candado recalcula el margen de las unidades vendidas **con la mecánica activa** — el peor caso — usando el descuento efectivo real (que incluye los redondeos y la tolerancia): $m_d = 1-(1-m)/(1-d)$. Cualquier oferta bajo el 22% se degrada con motivo. Nótese que el piso se exige sobre $m_d$, no sobre el margen mezclado de la sección 11 (que siempre es mayor): la regla protege **cada unidad vendida en promoción**, no un promedio.

In [10]:
df['MARGEN_OFERTA'] = (1 - (1 - m) / (1 - df['DESC_EFECTIVO'])) * 100

viola = (df['MECANICA'] != 'Sin oferta') & (df['MARGEN_OFERTA'] < PISO_MARGEN * 100)
df.loc[viola, ['ESTRATEGIA', 'DESC_EFECTIVO']] = ['Sin oferta', 0.0]
df.loc[viola, 'PRECIO_OFERTA'] = df.loc[viola, 'PRECIO JUSTO']
df.loc[viola, 'MOTIVO_SIN_OFERTA'] = 'Degradada: violaría el piso de margen'
df['MECANICA'] = df['ESTRATEGIA'].str.split(':').str[0]
df['MARGEN_OFERTA'] = ((1 - (1 - m) / (1 - df['DESC_EFECTIVO'])) * 100).round(2)

con = df[df['MECANICA'] != 'Sin oferta']
print(f"Degradadas por piso: {viola.sum()} | Margen mínimo en promoción: {con['MARGEN_OFERTA'].min():.2f}%")

Degradadas por piso: 19 | Margen mínimo en promoción: 22.03%


## 11 · Modelo económico de la promoción (el corazón del rediseño)
Tres piezas, cada una con su fórmula y su porqué.

### Pieza 1 — La demanda responde a la oferta exhibida
$$u = |E| \times MULT \times d \qquad\qquad Q_1 = Q_0\,(1 + u)$$
$|E|$ es la elasticidad base del tag y $MULT$ el multiplicador promocional: una oferta comunicada en la app (badge, precio tachado, "3 x \$99") produce una respuesta mayor que un cambio silencioso de lista, por visibilidad y ancla psicológica. La demanda responde a la **profundidad exhibida** $d$ completa — el cliente ve "-15%", no el promedio mezclado.

### Pieza 2 — Pero el descuento solo lo paga quien lo activa
Con tasa de redención $r$ (fracción de unidades vendidas con la mecánica activa), el ingreso por unidad **promedio** es:
$$\bar{P} = r\,P(1-d) + (1-r)\,P = P\,(1 - r\,d)$$
Y la utilidad por unidad promedio tiene una forma igual de limpia. La unidad a precio lleno deja $P_{neto}\,m$; la unidad en promoción deja $P_{neto}(1-d) - P_{neto}(1-m) = P_{neto}(m-d)$ — el ingreso neto baja con el descuento, el costo no. Mezclando:
$$\bar{\pi} = r\,P_{neto}(m-d) + (1-r)\,P_{neto}\,m = P_{neto}\,(m - r\,d)$$

### Pieza 3 — Los totales del día de oferta
$$GMV_1 = Q_1 \cdot P\,(1-r\,d) \qquad\qquad \Pi_1 = Q_1 \cdot P_{neto}\,(m - r\,d)$$
contra la base $GMV_0 = Q_0 P$ y $\Pi_0 = Q_0 P_{neto} m$. De aquí sale la **condición de rentabilidad** que gobierna la sección 12: la utilidad incremental es positiva si y solo si

$$(1+u)\,(m - r\,d) > m \;\;\Longleftrightarrow\;\; u > \frac{r\,d}{m - r\,d}$$

Léase: el uplift debe superar el costo de la promoción **como fracción del margen que sobrevive**. Un SKU con margen 35%, descuento 15% y redención 55% necesita $u > 0{,}0825/0{,}2675 = 31\%$ de uplift para pagar su promoción — y con $|E| = 1{,}5$ y $MULT = 2$ proyecta $u = 45\%$: pasa con holgura. Uno poco elástico con el mismo costo rara vez pasa. El modelo hace visible este umbral por SKU en lugar de dejarlo implícito.

In [11]:
# Redención por tipo de mecánica: a mayor umbral de unidades, menor conversión
def redencion(mec, estrategia):
    """Devuelve la tasa de redención según la mecánica asignada."""
    if mec in ('4x3', '5x4', '6x5'):
        return REDENCION['multibuy_grande']
    if mec == 'BNSP':
        return REDENCION['pack_3']
    if mec in ('SPON', 'BNSDP'):
        return REDENCION['umbral_2']
    return 0.0

df['REDENCION'] = [redencion(mc, es) for mc, es in zip(df['MECANICA'], df['ESTRATEGIA'])]
df['ELASTICIDAD_SUPUESTA'] = df['TAG ELAS'].map(ELASTICIDAD_POR_TAG).fillna(0)
df['Q0_DIA'] = df['TAG_ROTACION'].map(Q0_POR_ROTACION).fillna(0)

# Uplift del escenario base: responde a la profundidad exhibida completa
d_eff = df['DESC_EFECTIVO']
df['UPLIFT'] = df['ELASTICIDAD_SUPUESTA'].abs() * MULT_PROMO * d_eff
df['Q1_DIA'] = df['Q0_DIA'] * (1 + df['UPLIFT'])

# Totales del día de oferta con precio y utilidad MEZCLADOS por redención
rd = df['REDENCION'] * d_eff
df['GMV_BASE_DIA']  = (df['Q0_DIA'] * df['PRECIO JUSTO']).round(2)
df['GMV_PROY_DIA']  = (df['Q1_DIA'] * df['PRECIO JUSTO'] * (1 - rd)).round(2)
df['UTIL_BASE_DIA'] = (df['Q0_DIA'] * df['PRECIO_NETO'] * m).round(2)
df['UTIL_PROY_DIA'] = (df['Q1_DIA'] * df['PRECIO_NETO'] * (m - rd)).round(2)
df['GMV_INC_DIA']   = (df['GMV_PROY_DIA']  - df['GMV_BASE_DIA']).round(2)
df['UTIL_INC_DIA']  = (df['UTIL_PROY_DIA'] - df['UTIL_BASE_DIA']).round(2)

ofer = df[df['MECANICA'] != 'Sin oferta']
print(f"Uplift promedio de unidades (ofertas): {ofer['UPLIFT'].mean()*100:.0f}%")
print(f"Descuento mezclado promedio (r x d):   {(ofer['REDENCION']*ofer['DESC_EFECTIVO']).mean()*100:.1f}% "
      f"(vs profundidad exhibida {ofer['DESC_EFECTIVO'].mean()*100:.1f}%)")

Uplift promedio de unidades (ofertas): 16%
Descuento mezclado promedio (r x d):   3.2% (vs profundidad exhibida 6.5%)


## 12 · Guardrail económico: solo ofertas con utilidad incremental positiva
El piso del 22% protege el margen **unitario**; este segundo guardrail protege el **resultado total**: toda oferta cuya utilidad incremental proyectada sea negativa en el escenario base se descarta con motivo. Es la traducción operativa de la condición $u > r\,d/(m - r\,d)$: el modelo no publica promociones que, según sus propios supuestos, destruyen utilidad. El resultado agregado queda positivo **por construcción**, y la discusión con el negocio se desplaza a donde debe estar: los supuestos ($MULT$, $r$, elasticidades), no el signo.

In [12]:
destruye = (df['MECANICA'] != 'Sin oferta') & (df['UTIL_INC_DIA'] < 0)
df.loc[destruye, ['ESTRATEGIA', 'DESC_EFECTIVO']] = ['Sin oferta', 0.0]
df.loc[destruye, 'PRECIO_OFERTA'] = df.loc[destruye, 'PRECIO JUSTO']
df.loc[destruye, 'MOTIVO_SIN_OFERTA'] = 'Descartada: utilidad incremental proyectada < 0'
df['MECANICA'] = df['ESTRATEGIA'].str.split(':').str[0]

# Se anulan las proyecciones de las descartadas (quedan en base) y se refresca el margen
anular = df['MECANICA'] == 'Sin oferta'
df.loc[anular, ['UPLIFT', 'REDENCION']] = 0.0
df.loc[anular, 'Q1_DIA'] = df.loc[anular, 'Q0_DIA']
df.loc[anular, 'GMV_PROY_DIA']  = df.loc[anular, 'GMV_BASE_DIA']
df.loc[anular, 'UTIL_PROY_DIA'] = df.loc[anular, 'UTIL_BASE_DIA']
df.loc[anular, ['GMV_INC_DIA', 'UTIL_INC_DIA']] = 0.0
df['MARGEN_OFERTA'] = ((1 - (1 - m) / (1 - df['DESC_EFECTIVO'])) * 100).round(2)

ofer = df[df['MECANICA'] != 'Sin oferta']
print(f"Descartadas por economía negativa: {destruye.sum()}")
print(f"Ofertas finales: {len(ofer)} | Utilidad incremental mínima por SKU: ${ofer['UTIL_INC_DIA'].min():,.2f}")

Descartadas por economía negativa: 161
Ofertas finales: 351 | Utilidad incremental mínima por SKU: $0.04


## 13 · Día de ejecución
Sin catálogo de departamentos en los insumos autorizados, la regla usa los atributos disponibles, en orden de precedencia:

1. **Tema Mundial → Sábado 4**: partidos de octavos, pico de consumo social.
2. **Alta rotación → Viernes 3**: arranque del fin de semana con los productos de mayor tráfico, que llenan carritos desde el primer día.
3. **BNSDP (ticket alto) → Domingo 5**: la compra de reposición y ticket grande se decide con más calma; el domingo captura la planeación de la semana entrante.
4. **Resto → Sábado 4**: el día insignia del fin de semana.

In [13]:
def dia_ejecucion(r):
    """Asigna el día del FDS a cada oferta según su perfil."""
    if r['MECANICA'] == 'Sin oferta':
        return ''
    if r['TEMA_MUNDIAL']:
        return 'Sábado'
    if r['TAG_ROTACION'] == 'Alta rotacion':
        return 'Viernes'
    if r['MECANICA'] == 'BNSDP':
        return 'Domingo'
    return 'Sábado'

df['DIA_EJECUCION'] = df.apply(dia_ejecucion, axis=1)
print(df[df['MECANICA'] != 'Sin oferta']['DIA_EJECUCION'].value_counts().to_string())

DIA_EJECUCION
Sábado     176
Viernes    137
Domingo     38


## 14 · Escenarios de sensibilidad
El multiplicador promocional es el supuesto con más varianza real, así que se somete a sensibilidad: el uplift de cada oferta se escala por 0,6× (Conservador), 1,0× (Base) y 1,4× (Optimista) y se recalculan los totales con las mismas identidades mezcladas. La lectura correcta del cuadro: si incluso el escenario conservador deja utilidad incremental positiva, la estrategia es robusta al supuesto más frágil del modelo.

In [14]:
filas = []
o = df[df['MECANICA'] != 'Sin oferta']
rd_o = o['REDENCION'] * o['DESC_EFECTIVO']
for nombre, f in ESCENARIOS.items():
    q1 = o['Q0_DIA'] * (1 + o['UPLIFT'] * f)
    gmv1  = (q1 * o['PRECIO JUSTO'] * (1 - rd_o)).sum()
    util1 = (q1 * o['PRECIO_NETO'] * (o['MARGEN']/100 - rd_o)).sum()
    filas.append({'Escenario': nombre, 'Factor uplift': f,
                  'Unidades': round(q1.sum()), 'GMV día oferta': round(gmv1),
                  'GMV incremental': round(gmv1 - o['GMV_BASE_DIA'].sum()),
                  'Utilidad incremental': round(util1 - o['UTIL_BASE_DIA'].sum())})
escenarios_df = pd.DataFrame(filas)
print(f"Base de comparación (mismas ofertas sin promo): unidades {o['Q0_DIA'].sum():,.0f} | "
      f"GMV ${o['GMV_BASE_DIA'].sum():,.0f} | utilidad ${o['UTIL_BASE_DIA'].sum():,.0f}")
escenarios_df

Base de comparación (mismas ofertas sin promo): unidades 3,188 | GMV $223,592 | utilidad $62,875


,Escenario,Factor uplift,Unidades,GMV día oferta,GMV incremental,Utilidad incremental
0,Conservador,0.6,3568,243983,20392,1000
1,Base,1.0,3821,262806,39214,6312
2,Optimista,1.4,4074,281628,58036,11624


## 15 · Resultados finales

In [15]:
ofer = df[df['MECANICA'] != 'Sin oferta']

print("=== ESTRATEGIA ===")
print(df['MECANICA'].value_counts().to_string())
print(f"\nOfertas: {len(ofer)} de {len(df)} ({len(ofer)/len(df)*100:.1f}%) | "
      f"descuento exhibido promedio: {ofer['DESC_EFECTIVO'].mean()*100:.1f}% | "
      f"margen mínimo en promoción: {ofer['MARGEN_OFERTA'].min():.2f}%")

print("\n=== PROYECCIÓN ESCENARIO BASE (día de oferta de cada SKU) ===")
print(f"Unidades: {ofer['Q0_DIA'].sum():,.0f} -> {ofer['Q1_DIA'].sum():,.0f} "
      f"({(ofer['Q1_DIA'].sum()/ofer['Q0_DIA'].sum()-1)*100:+.1f}%)")
print(f"GMV:      ${ofer['GMV_BASE_DIA'].sum():,.0f} -> ${ofer['GMV_PROY_DIA'].sum():,.0f} "
      f"({(ofer['GMV_PROY_DIA'].sum()/ofer['GMV_BASE_DIA'].sum()-1)*100:+.1f}%)")
print(f"Utilidad: ${ofer['UTIL_BASE_DIA'].sum():,.0f} -> ${ofer['UTIL_PROY_DIA'].sum():,.0f} "
      f"({(ofer['UTIL_PROY_DIA'].sum()/ofer['UTIL_BASE_DIA'].sum()-1)*100:+.1f}%)")

print("\n=== POR DÍA ===")
print(ofer.groupby('DIA_EJECUCION').agg(Ofertas=('SKU','count'),
      GMV_inc=('GMV_INC_DIA','sum'), Util_inc=('UTIL_INC_DIA','sum')).round(0).to_string())

mun = ofer[ofer['TEMA_MUNDIAL']]
print(f"\n=== MUNDIAL ===\nOfertas: {len(mun)} | GMV incremental: ${mun['GMV_INC_DIA'].sum():,.0f} | "
      f"Utilidad incremental: ${mun['UTIL_INC_DIA'].sum():,.0f}")

print("\n=== TOP 10 POR UTILIDAD INCREMENTAL ===")
cols = ['STORE_ID','Nombre','MARGEN','ESTRATEGIA','DIA_EJECUCION','TEMA_MUNDIAL',
        'UPLIFT','GMV_INC_DIA','UTIL_INC_DIA','MARGEN_OFERTA']
ofer.nlargest(10, 'UTIL_INC_DIA')[cols]

=== ESTRATEGIA ===
MECANICA
Sin oferta    2121
SPON           192
BNSP            93
BNSDP           44
6x5             16
5x4              4
4x3              2

Ofertas: 351 de 2472 (14.2%) | descuento exhibido promedio: 6.7% | margen mínimo en promoción: 22.03%

=== PROYECCIÓN ESCENARIO BASE (día de oferta de cada SKU) ===
Unidades: 3,188 -> 3,821 (+19.8%)
GMV:      $223,592 -> $262,806 (+17.5%)
Utilidad: $62,875 -> $69,187 (+10.0%)

=== POR DÍA ===
               Ofertas  GMV_inc  Util_inc
DIA_EJECUCION                            
Domingo             38   2019.0     212.0
Sábado             176   8442.0    2223.0
Viernes            137  28753.0    3877.0

=== MUNDIAL ===
Ofertas: 17 | GMV incremental: $5,608 | Utilidad incremental: $1,789

=== TOP 10 POR UTILIDAD INCREMENTAL ===


,STORE_ID,Nombre,MARGEN,ESTRATEGIA,DIA_EJECUCION,TEMA_MUNDIAL,UPLIFT,GMV_INC_DIA,UTIL_INC_DIA,MARGEN_OFERTA
425,9,Boneless Búfalo Bachoco 700g,46.85,4x3,Sábado,True,0.75000,2639.02,876.54,29.13
692,14,Boneless Búfalo Bachoco 700g,46.85,4x3,Sábado,True,0.75000,2639.02,876.54,29.13
1331,9,Camarón Grande 21-25 La Sanitaria Congelado 400g,41.68,5x4,Viernes,False,0.60000,1707.02,483.00,27.10
2246,14,Camarón Grande 21-25 La Sanitaria Congelado 400g,41.68,5x4,Viernes,False,0.60000,1707.02,483.00,27.10
63,9,Acondicionador Cantu Aguacate y Manteca de Karité 400ml,35.04,6x5,Viernes,False,0.73348,1857.94,394.81,22.04
1733,14,Licor de Hierbas Fernet Branca 750ml,26.88,"BNSDP: 2 uds, ahorra $25 c/u",Viernes,False,0.21912,1864.38,144.27,23.05
2152,14,Shampoo Herbal Essences Bio Renew Coconut Milk 400ml,34.04,"BNSDP: 2 uds, ahorra $25 c/u",Viernes,False,0.42570,1107.74,102.16,23.13
351,9,Jugo Jumex Único Fresco Manzana 960ml,37.08,6x5,Viernes,False,0.50010,394.18,93.55,24.49
357,9,Chocolate Amargo Lindt Excellence 70% Cacao 100g,37.48,6x5,Viernes,False,0.26672,586.61,73.41,24.97
630,14,Chocolate Amargo Lindt Excellence 70% Cacao 100g,37.48,6x5,Viernes,False,0.26672,586.61,73.41,24.97


**Interpretación de los resultados finales.**

- **El rediseño produce números con sentido económico.** En el escenario base, las 351 ofertas proyectan unidades +19,8%, GMV +17,5% y utilidad +10,0% en sus días de ejecución. La diferencia contra un modelo ingenuo está en una sola cifra: el descuento **mezclado** que realmente paga la venta es 3,2%, la mitad de la profundidad exhibida de 6,7% — porque solo la fracción redimida (35–55% según mecánica) compra al precio rebajado, mientras la demanda responde a la profundidad completa que ve en la app.
- **El guardrail económico descartó 161 candidatas** — en su mayoría SKUs poco elásticos cuyo uplift proyectado no alcanzaba a pagar el costo mezclado de su promoción ($u < r\,d/(m-r\,d)$). La cobertura baja de 20,7% a 14,2%, y ese recorte es exactamente lo que garantiza que **cada oferta publicada deja más utilidad de la que cuesta** (mínima incremental por SKU: \$0,04; total positivo por construcción).
- **La sensibilidad sostiene la decisión:** incluso en el escenario conservador (uplift al 60% del base) la utilidad incremental del portafolio se mantiene positiva, porque el guardrail exige holgura en el escenario base y la mezcla por redención amortigua el costo. Si el multiplicador promocional real resultara aún menor, el primer efecto sería utilidad incremental cercana a cero, no destrucción — el diseño falla hacia el lado seguro.
- **Los dos guardrails cumplen papeles distintos y ambos quedaron verificados:** el piso protege cada unidad vendida en promoción (margen mínimo 22,03%) y el económico protege el total (ninguna oferta con incremental negativo). La discusión con el negocio queda donde debe: en los supuestos declarados ($MULT$, $r$, elasticidades, $Q_0$), no en el signo del resultado.
- **Validación de los cruces:** blacklist con cero coincidencias (el pre-filtrado de la fuente operó) y 7 filas protegidas de doble descuento por campañas que inician durante el FDS — el punto ciego que este cruce existe para cubrir.

## 16 · Exportación del Excel final
Cuatro hojas: `Estrategia SKU` (las 2.472 filas — cada SKU del universo con su decisión y su porqué; verde = oferta, dorado = oferta Mundial), `Resumen` (indicadores globales, por día y por mecánica), `Escenarios` (la tabla de sensibilidad) y `Leyenda` (diccionario de columnas para leer el archivo sin abrir el notebook).

In [16]:
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

cols_out = ['STORE_ID','SKU','Nombre','COSTO','IEPS','IVA','MARGEN','PRECIO JUSTO','TAG_MARGEN',
            'INDEX PRECIO DCTO','PRECIO COMP DCTO','PRECIO COMP LINEA','INDEX PRECIO LINEA',
            'TAG_ROTACION','TAG_SHARE','TAG ELAS','PROMO ACTIVA HOY','TEMA_MUNDIAL',
            'MOTIVO_SIN_OFERTA','ESTRATEGIA','DIA_EJECUCION','PRECIO_OFERTA','DESC_EFECTIVO',
            'MARGEN_OFERTA','REDENCION','ELASTICIDAD_SUPUESTA','UPLIFT','Q0_DIA','Q1_DIA',
            'GMV_BASE_DIA','GMV_PROY_DIA','GMV_INC_DIA','UTIL_BASE_DIA','UTIL_PROY_DIA','UTIL_INC_DIA']
out = df[cols_out].copy()
out['DESC_EFECTIVO'] = (out['DESC_EFECTIVO']*100).round(1)
out['UPLIFT'] = (out['UPLIFT']*100).round(0)
out = out.rename(columns={'DESC_EFECTIVO':'DESC_EFECTIVO_%','MARGEN_OFERTA':'MARGEN_OFERTA_%',
                          'UPLIFT':'UPLIFT_%'})
out = out.sort_values(['DIA_EJECUCION','STORE_ID','ESTRATEGIA']).reset_index(drop=True)

resumen = pd.DataFrame({
    'Indicador': ['Fin de semana objetivo','Universo (tienda x SKU)','Ofertas asignadas','% cobertura',
                  'Descuento exhibido promedio','Margen mínimo en promoción',
                  'Ofertas Tema Mundial','Unidades base -> proyectadas (escenario base)',
                  'GMV base -> proyectado (escenario base)','Utilidad base -> proyectada (escenario base)',
                  'Utilidad incremental total (base)','Guardrails aplicados'],
    'Valor': [f"{WEEKEND_INICIO.date()} a {WEEKEND_FIN.date()}", len(df), len(ofer),
              f"{len(ofer)/len(df)*100:.1f}%", f"{ofer['DESC_EFECTIVO'].mean()*100:.1f}%",
              f"{ofer['MARGEN_OFERTA'].min():.2f}%", int(ofer['TEMA_MUNDIAL'].sum()),
              f"{ofer['Q0_DIA'].sum():,.0f} -> {ofer['Q1_DIA'].sum():,.0f}",
              f"${ofer['GMV_BASE_DIA'].sum():,.0f} -> ${ofer['GMV_PROY_DIA'].sum():,.0f}",
              f"${ofer['UTIL_BASE_DIA'].sum():,.0f} -> ${ofer['UTIL_PROY_DIA'].sum():,.0f}",
              f"${ofer['UTIL_INC_DIA'].sum():,.0f}",
              'Piso 22% por unidad en promo + utilidad incremental >= 0 por SKU']
})
por_dia = ofer.groupby('DIA_EJECUCION').agg(Ofertas=('SKU','count'),
          GMV_incremental=('GMV_INC_DIA','sum'), Utilidad_incremental=('UTIL_INC_DIA','sum')).round(0).reset_index()
mecs = df['MECANICA'].value_counts().rename_axis('Mecanica').reset_index(name='SKU_tienda')

leyenda = pd.DataFrame([
    ('MOTIVO_SIN_OFERTA','Por qué la fila no lleva oferta (vacío = lleva oferta)'),
    ('TEMA_MUNDIAL','True si el nombre coincide con el patrón de consumo "ver el partido"'),
    ('ESTRATEGIA / DIA_EJECUCION','Mecánica asignada y día del FDS en que corre (Vie 3 / Sáb 4 / Dom 5 jul)'),
    ('PRECIO_OFERTA','Precio por unidad cuando la mecánica se activa (= góndola si Sin oferta)'),
    ('DESC_EFECTIVO_% / MARGEN_OFERTA_%','Profundidad exhibida y pure margin de las unidades en promoción'),
    ('REDENCION','Fracción supuesta de unidades que se venden con la mecánica activa'),
    ('ELASTICIDAD_SUPUESTA / UPLIFT_%','Elasticidad base por tag y crecimiento proyectado de unidades'),
    ('Q0_DIA / Q1_DIA','Unidades/día sin y con promoción (Q0 es supuesto de dimensionamiento por rotación)'),
    ('GMV_*_DIA','Venta en pesos del día de oferta: base, proyectada (precio mezclado) e incremental'),
    ('UTIL_*_DIA','Pure margin en pesos del día de oferta: base, proyectada (mezclada) e incremental'),
], columns=['Columna','Descripción'])

with pd.ExcelWriter(RUTA_SALIDA, engine='openpyxl') as w:
    out.to_excel(w, sheet_name='Estrategia SKU', index=False)
    resumen.to_excel(w, sheet_name='Resumen', index=False, startrow=0)
    por_dia.to_excel(w, sheet_name='Resumen', index=False, startrow=len(resumen)+3)
    mecs.to_excel(w, sheet_name='Resumen', index=False, startrow=len(resumen)+3+len(por_dia)+3)
    escenarios_df.to_excel(w, sheet_name='Escenarios', index=False)
    leyenda.to_excel(w, sheet_name='Leyenda', index=False)

    wb = w.book; ws = wb['Estrategia SKU']
    hdr = PatternFill('solid', start_color='1F4E79')
    for c in ws[1]:
        c.font = Font(bold=True, color='FFFFFF', name='Arial', size=10)
        c.fill = hdr; c.alignment = Alignment(horizontal='center', vertical='center')
    ws.freeze_panes = 'D2'; ws.auto_filter.ref = ws.dimensions
    verde, dorado = PatternFill('solid', start_color='E2EFDA'), PatternFill('solid', start_color='FFF2CC')
    i_est = [c.value for c in ws[1]].index('ESTRATEGIA'); i_mun = [c.value for c in ws[1]].index('TEMA_MUNDIAL')
    for row in ws.iter_rows(min_row=2):
        if row[i_est].value != 'Sin oferta':
            f = dorado if row[i_mun].value else verde
            for cell in row: cell.fill = f
    for i in range(1, ws.max_column + 1):
        ws.column_dimensions[get_column_letter(i)].width = {3: 50}.get(i, 14)
    for hoja in ['Resumen','Escenarios','Leyenda']:
        ws2 = wb[hoja]
        for c in ws2[1]:
            c.font = Font(bold=True, color='FFFFFF', name='Arial', size=10); c.fill = hdr
        ws2.column_dimensions['A'].width = 46; ws2.column_dimensions['B'].width = 56

print(f"Exportado: {RUTA_SALIDA} | {len(out)} filas | {len(ofer)} con oferta "
      f"({int(ofer['TEMA_MUNDIAL'].sum())} Mundial)")

Exportado: Estrategia_Pricing_FDS_Julio.xlsx | 2472 filas | 351 con oferta (17 Mundial)


---
## Cierre y supuestos declarados
**Modelo:** la demanda responde a la profundidad exhibida amplificada por el efecto promocional ($u = |E| \cdot MULT \cdot d$); el costo de la promoción lo paga solo la fracción redimida ($\bar{P} = P(1-rd)$, $\bar{\pi} = P_{neto}(m-rd)$); y ninguna oferta se publica sin utilidad incremental proyectada positiva.

**Supuestos, todos en la celda de parámetros:** elasticidades por tag ($-2{,}2 / -1{,}5 / -0{,}8$), multiplicador promocional 2,0 (con sensibilidad 0,6×–1,4×), redención por mecánica (0,55 / 0,40 / 0,35 según umbral) y $Q_0$ por rotación como dimensionamiento — los porcentajes por SKU no dependen de $Q_0$; los pesos totales escalan con él. La validación definitiva es la medición post-FDS contra estas mismas proyecciones, SKU por SKU y día por día.

**En una frase:** *la estrategia financia cada oferta con el colchón de margen del propio SKU, la dirige a donde elasticidad, rotación y la temporada Mundial prometen respuesta, la comunica con mecánicas de volumen que solo descuentan a quien las activa, la agenda en el día correcto del fin de semana, y solo publica ofertas que — bajo supuestos declarados y sensibilizados — dejan más utilidad de la que cuestan.*